# Phoenix14T Pipeline Pretrain

Self-contained Kaggle notebook for skeleton-only continuous sign language recognition. The notebook is set to `RUN_MODE = "full"` for training. Change it to `"smoke"` only when you need a quick sanity check.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import math
import os
import random
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [2]:
# Run mode:
#   full : full training run.
#   smoke: small subset for path/model sanity check if you need to re-check paths.
RUN_MODE = "full"

EXPERIMENT_NAME = "phoenix14t_pipeline_pretrain"
MODEL_NAME = "pipeline"
STAGE = "pretrain"
DATASET_NAME = "PHOENIX14T"

DATA_DIR = Path(r"/kaggle/input/datasets/smartstu/skeleton/skeleton")
META_PATH = DATA_DIR / "dataset_meta.json"
OUTPUT_DIR = Path("/kaggle/working") / EXPERIMENT_NAME
PRETRAIN_CKPT_PATH = Path(r"")

CONFIG = {
    "seed": 491,
    "batch_size": 8,
    "accum_steps": 2,
    "epochs": 100,
    "lr": 0.0004,
    "weight_decay": 0.0001,
    "warmup_epochs": 5,
    "patience": 12,
    "grad_clip": 5.0,
    "num_workers": 2,
    "hidden": 256,
    "dropout": 0.25,
    "adaptive_gcn": True,
    "aux_weight": 0.3,
    "lambda_kl": 0.05,
    "kl_temp": 1.0,
    "use_motion": True,
    "use_conf_gate": True,
    "num_blocks": 3,
    "num_heads": 8,
    "stream_weight": 1.0,
    "stream_ramp_epochs": 20,
    "distill_warmup": 25,
    "distill_temp": 2.0,
    "distill_max_weight": 0.15,
    "distill_step": 0.015,
    "root_normalize": False,
    "freeze_backbone_epochs": 0,
    "time_stretch_prob": 0.6,
    "time_stretch_range": (0.85, 1.15),
    "spatial_jitter_std": 0.01,
    "confidence_dropout_prob": 0.1,
    "confidence_dropout_ratio": 0.05,
    "enable_horizontal_flip": False,
    "horizontal_flip_prob": 0.0,
    "beam_width": 8,
    "beam_topk": 25,
    "run_beam_in_smoke": False,
    "vocab_source": "all",
    "strict_oov_check": True,
    "smoke_train_samples": 32,
    "smoke_eval_samples": 8,
    "smoke_epochs": 2,
}

if RUN_MODE == "smoke":
    CONFIG["epochs"] = CONFIG["smoke_epochs"]
    CONFIG["patience"] = CONFIG["smoke_epochs"]
    CONFIG["num_workers"] = 0

print("Experiment:", EXPERIMENT_NAME)
print("Data dir:", DATA_DIR)
print("Metadata:", META_PATH)
print("Output dir:", OUTPUT_DIR)
print("Run mode:", RUN_MODE)


Experiment: phoenix14t_pipeline_pretrain
Data dir: /kaggle/input/datasets/smartstu/skeleton/skeleton
Metadata: /kaggle/input/datasets/smartstu/skeleton/skeleton/dataset_meta.json
Output dir: /kaggle/working/phoenix14t_pipeline_pretrain
Run mode: full


## Dataset and vocabulary

Vocabulary construction is controlled by `CONFIG["vocab_source"]`. VieCSL uses train-only vocabulary with strict OOV checking; Phoenix14T pretraining uses all official annotations because Phoenix dev/test contain a small number of glosses that do not appear in train.


In [3]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_metadata(meta_path: Path):
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    if not isinstance(meta, list):
        raise ValueError("Metadata must be a list of sample dictionaries.")
    required = {"video", "split", "gloss"}
    missing = [i for i, item in enumerate(meta) if not required.issubset(item)]
    if missing:
        raise ValueError(f"Metadata records missing keys at indices: {missing[:10]}")
    return meta


def build_vocab(meta, source: str = "train"):
    counter = Counter()
    for item in meta:
        if source == "all" or item["split"] == "train":
            counter.update(item["gloss"].split())
    if not counter:
        raise ValueError(f"No tokens found while building vocabulary from source={source!r}.")
    tokens = sorted(counter.keys(), key=lambda w: (-counter[w], w))
    vocab = {"<blank>": 0}
    for idx, token in enumerate(tokens, 1):
        vocab[token] = idx
    inv_vocab = {idx: token for token, idx in vocab.items()}
    return vocab, inv_vocab, counter


def collect_oov(meta, vocab):
    oov_by_split = defaultdict(set)
    for item in meta:
        for token in item["gloss"].split():
            if token not in vocab:
                oov_by_split[item["split"]].add(token)
    return {k: sorted(v) for k, v in oov_by_split.items()}


def assert_no_oov(meta, vocab):
    oov_by_split = collect_oov(meta, vocab)
    if oov_by_split:
        print("OOV tokens found:", oov_by_split)
        raise ValueError("Dev/test contains OOV tokens. Fix split or vocabulary first.")


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_csv(rows, path: Path, fieldnames=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        keys = set()
        for row in rows:
            keys.update(row.keys())
        fieldnames = sorted(keys)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def resolve_npy_path(data_dir: Path, item: dict):
    rel = item.get("npy_path")
    if rel:
        path = data_dir / rel
        if path.exists():
            return path
    return data_dir / item["split"] / f"{item['video']}.npy"


class SkeletonDataset(Dataset):
    def __init__(self, meta, vocab, split: str, augment: bool):
        self.items = [item for item in meta if item["split"] == split]
        self.vocab = vocab
        self.split = split
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def encode(self, gloss: str):
        return [self.vocab[token] for token in gloss.split()]

    @staticmethod
    def compute_motion(sk: np.ndarray):
        xy = sk[:, :, :2]
        dx_prev = np.concatenate([np.zeros_like(xy[:1]), xy[1:] - xy[:-1]], axis=0)
        dx_next = np.concatenate([xy[1:] - xy[:-1], np.zeros_like(xy[:1])], axis=0)
        motion = np.concatenate([dx_prev, dx_next], axis=-1)
        return np.clip(motion, -1.0, 1.0).astype(np.float32)

    @staticmethod
    def time_resample(arr: np.ndarray, new_len: int):
        if len(arr) == new_len:
            return arr
        idx = np.linspace(0, len(arr) - 1, new_len).astype(np.int64)
        return arr[idx]

    def augment_skeleton(self, sk: np.ndarray):
        if random.random() < CONFIG["time_stretch_prob"]:
            new_len = max(8, int(len(sk) * random.uniform(*CONFIG["time_stretch_range"])))
            sk = self.time_resample(sk, new_len)
        if CONFIG["spatial_jitter_std"] > 0:
            sk[:, :, :2] += np.random.randn(*sk[:, :, :2].shape).astype(np.float32) * CONFIG["spatial_jitter_std"]
        if random.random() < CONFIG["confidence_dropout_prob"]:
            drop = np.random.rand(sk.shape[0], sk.shape[1], 1) < CONFIG["confidence_dropout_ratio"]
            sk[:, :, 2:3] = np.where(drop, 0.0, sk[:, :, 2:3])
        if CONFIG["enable_horizontal_flip"] and random.random() < CONFIG["horizontal_flip_prob"]:
            sk[:, :, 0] = -sk[:, :, 0]
            sk_new = sk.copy()
            sk_new[:, 33:54, :] = sk[:, 54:75, :]
            sk_new[:, 54:75, :] = sk[:, 33:54, :]
            sk = sk_new
        return sk

    def __getitem__(self, idx):
        item = self.items[idx]
        path = resolve_npy_path(DATA_DIR, item)
        sk = np.load(path, allow_pickle=False).astype(np.float32)
        if sk.ndim != 3 or sk.shape[1] != 86 or sk.shape[2] != 3:
            raise ValueError(f"Expected skeleton shape (T, 86, 3), got {sk.shape} for {path}")
        sk = np.nan_to_num(sk, nan=0.0, posinf=0.0, neginf=0.0)
        if CONFIG["root_normalize"]:
            root_xy = sk[:, 0:1, :2].copy()
            sk[:, :, :2] = sk[:, :, :2] - root_xy
        if self.augment:
            sk = self.augment_skeleton(sk)

        motion = None
        if MODEL_NAME == "pipeline":
            motion = self.compute_motion(sk)
        elif CONFIG["use_motion"]:
            motion_xy = self.compute_motion(sk)[:, :, :2]
            sk = np.concatenate([sk, motion_xy], axis=-1).astype(np.float32)

        y = torch.tensor(self.encode(item["gloss"]), dtype=torch.long)
        sample = {
            "sk": torch.from_numpy(sk.astype(np.float32)),
            "mo": None if motion is None else torch.from_numpy(motion),
            "y": y,
            "input_len": int(sk.shape[0]),
            "target_len": int(len(y)),
            "item": item,
        }
        return sample


def collate_fn(batch):
    sk = pad_sequence([b["sk"] for b in batch], batch_first=True, padding_value=0.0)
    if batch[0]["mo"] is None:
        mo = None
    else:
        mo = pad_sequence([b["mo"] for b in batch], batch_first=True, padding_value=0.0)
    y = torch.cat([b["y"] for b in batch], dim=0)
    input_lens = torch.tensor([b["input_len"] for b in batch], dtype=torch.long)
    target_lens = torch.tensor([b["target_len"] for b in batch], dtype=torch.long)
    items = [b["item"] for b in batch]
    return sk, mo, y, input_lens, target_lens, items


set_seed(CONFIG["seed"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

meta = load_metadata(META_PATH)
vocab, inv_vocab, vocab_counter = build_vocab(meta, source=CONFIG["vocab_source"])
oov_report = collect_oov(meta, vocab)
if CONFIG["strict_oov_check"]:
    assert_no_oov(meta, vocab)
elif oov_report:
    print("OOV tokens are present but allowed by config:", oov_report)

split_counts = Counter(item["split"] for item in meta)
print("Split counts:", dict(split_counts))
print("Vocabulary size:", len(vocab), "(including blank)")
print("Vocabulary source:", CONFIG["vocab_source"])
print("Vocabulary token count:", sum(vocab_counter.values()))
if oov_report:
    print("OOV report:", oov_report)

save_json(vocab, OUTPUT_DIR / "vocab.json")
save_json(CONFIG, OUTPUT_DIR / "config.json")
save_json(oov_report, OUTPUT_DIR / "oov_report.json")

train_ds = SkeletonDataset(meta, vocab, "train", augment=True)
dev_ds = SkeletonDataset(meta, vocab, "dev", augment=False)
test_ds = SkeletonDataset(meta, vocab, "test", augment=False)

if RUN_MODE == "smoke":
    train_ds = Subset(train_ds, range(min(CONFIG["smoke_train_samples"], len(train_ds))))
    dev_ds = Subset(dev_ds, range(min(CONFIG["smoke_eval_samples"], len(dev_ds))))
    test_ds = Subset(test_ds, range(min(CONFIG["smoke_eval_samples"], len(test_ds))))

loader_kwargs = dict(
    batch_size=CONFIG["batch_size"],
    collate_fn=collate_fn,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **loader_kwargs)
dev_loader = DataLoader(dev_ds, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **loader_kwargs)

print(f"Train={len(train_ds)} Dev={len(dev_ds)} Test={len(test_ds)}")


Split counts: {'train': 7096, 'dev': 519, 'test': 642}
Vocabulary size: 1116 (including blank)
Vocabulary source: all
Vocabulary token count: 63259
Train=7096 Dev=519 Test=642


## Model definition


In [4]:
def build_adjacency(num_nodes: int, edges: list[tuple[int, int]]):
    A = np.zeros((2, num_nodes, num_nodes), dtype=np.float32)
    A[0] = np.eye(num_nodes, dtype=np.float32)
    for i, j in edges:
        if 0 <= i < num_nodes and 0 <= j < num_nodes:
            A[1, i, j] = 1.0
            A[1, j, i] = 1.0
    for k in range(A.shape[0]):
        deg = A[k].sum(axis=1)
        deg[deg == 0] = 1.0
        D = np.diag(1.0 / np.sqrt(deg))
        A[k] = D @ A[k] @ D
    return torch.tensor(A, dtype=torch.float32)


EDGES_BODY = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
    (27,29),(28,30),(29,31),(30,32),(27,31),(28,32),
]
EDGES_HAND = [
    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),
    (13,17),(0,17),(17,18),(18,19),(19,20),
]
EDGES_FACE = [(0,2),(1,3),(2,4),(3,4),(4,5),(4,6),(0,1)]
EDGES_MOUTH = [(0,1),(1,2),(2,3),(3,0)]


class STGCNBlock(nn.Module):
    def __init__(self, in_c: int, out_c: int, num_nodes: int, edges: list, adaptive: bool):
        super().__init__()
        A = build_adjacency(num_nodes, edges)
        if adaptive:
            self.A = nn.Parameter(A.clone())
        else:
            self.register_buffer("A", A)
        self.spatial = nn.Conv2d(in_c * 2, out_c, kernel_size=1, bias=False)
        self.temporal = nn.Sequential(
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=(3, 1), padding=(1, 0), groups=out_c, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.residual = nn.Identity() if in_c == out_c else nn.Conv2d(in_c, out_c, kernel_size=1, bias=False)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (B, T, V, C)
        x = x.permute(0, 3, 1, 2).contiguous()
        parts = [torch.einsum("bctv,vw->bctw", x, self.A[k]) for k in range(2)]
        y = self.spatial(torch.cat(parts, dim=1))
        y = self.temporal(y) + self.residual(x)
        y = self.act(y)
        return y.permute(0, 2, 3, 1).contiguous()


class StreamEncoder(nn.Module):
    def __init__(self, in_c: int, hidden: int, num_nodes: int, edges: list, adaptive: bool):
        super().__init__()
        mid = max(64, hidden)
        self.blocks = nn.Sequential(
            STGCNBlock(in_c, 64, num_nodes, edges, adaptive),
            STGCNBlock(64, mid, num_nodes, edges, adaptive),
            STGCNBlock(mid, hidden, num_nodes, edges, adaptive),
        )

    def forward(self, x):
        x = self.blocks(x)
        return x.mean(dim=2)


def center_xy(x, root_idx: int):
    out = x.clone()
    out[:, :, :, :2] = out[:, :, :, :2] - out[:, :, root_idx:root_idx + 1, :2]
    return out


class PipelineCSLRModel(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        part_hidden = CONFIG["hidden"] // 4
        adaptive = CONFIG["adaptive_gcn"]
        self.body_sk = StreamEncoder(3, part_hidden, 33, EDGES_BODY, adaptive)
        self.lh_sk = StreamEncoder(3, part_hidden, 21, EDGES_HAND, adaptive)
        self.rh_sk = StreamEncoder(3, part_hidden, 21, EDGES_HAND, adaptive)
        self.face_sk = StreamEncoder(3, part_hidden, 7, EDGES_FACE, adaptive)
        self.mouth_sk = StreamEncoder(3, part_hidden, 4, EDGES_MOUTH, adaptive)
        self.body_mo = StreamEncoder(4, part_hidden, 33, EDGES_BODY, adaptive)
        self.lh_mo = StreamEncoder(4, part_hidden, 21, EDGES_HAND, adaptive)
        self.rh_mo = StreamEncoder(4, part_hidden, 21, EDGES_HAND, adaptive)
        self.face_mo = StreamEncoder(4, part_hidden, 7, EDGES_FACE, adaptive)
        self.mouth_mo = StreamEncoder(4, part_hidden, 4, EDGES_MOUTH, adaptive)

        feat_dim = part_hidden * 10
        self.temporal = nn.Sequential(
            nn.Conv1d(feat_dim, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(CONFIG["dropout"]),
            nn.Conv1d(256, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, ceil_mode=True),
        )
        self.lstm = nn.LSTM(256, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.cls_aux = nn.Linear(256, vocab_size)
        self.cls_pri = nn.Linear(512, vocab_size)

    def extract_features(self, sk, mo):
        body = center_xy(sk[:, :, 0:33, :], 0)
        lh = center_xy(sk[:, :, 33:54, :], 0)
        rh = center_xy(sk[:, :, 54:75, :], 0)
        face = center_xy(sk[:, :, 75:82, :], 4)
        mouth = center_xy(sk[:, :, 82:86, :], 0)

        body_m = mo[:, :, 0:33, :]
        lh_m = mo[:, :, 33:54, :]
        rh_m = mo[:, :, 54:75, :]
        face_m = mo[:, :, 75:82, :]
        mouth_m = mo[:, :, 82:86, :]

        return torch.cat([
            self.body_sk(body), self.lh_sk(lh), self.rh_sk(rh), self.face_sk(face), self.mouth_sk(mouth),
            self.body_mo(body_m), self.lh_mo(lh_m), self.rh_mo(rh_m), self.face_mo(face_m), self.mouth_mo(mouth_m),
        ], dim=-1)

    def forward(self, sk, mo):
        feat = self.extract_features(sk, mo)
        z = self.temporal(feat.transpose(1, 2)).transpose(1, 2)
        aux = self.cls_aux(z)
        z, _ = self.lstm(z)
        pri = self.cls_pri(z)
        return aux, pri


def create_model(vocab_size: int):
    return PipelineCSLRModel(vocab_size)


## Training, decoding, and evaluation utilities

The notebook saves local artifacts for the report: curves, WER breakdowns, prediction CSV files, and final metrics JSON.


In [5]:
def pooled_lengths(input_lens: torch.Tensor, output_time: int):
    lens = (input_lens + 1) // 2
    return lens.clamp(min=1, max=output_time).cpu()


def ctc_loss_for_logits(logits, targets, input_lens, target_lens, ctc_fn):
    feat_lens = pooled_lengths(input_lens, logits.shape[1])
    log_probs = logits.log_softmax(-1).permute(1, 0, 2)
    return ctc_fn(log_probs, targets, feat_lens, target_lens.cpu())


def symmetric_kl(a, b, valid_lens, temperature: float):
    log_a = F.log_softmax(a / temperature, dim=-1)
    log_b = F.log_softmax(b / temperature, dim=-1)
    prob_a = log_a.exp()
    prob_b = log_b.exp()
    kl_ab = F.kl_div(log_a, prob_b.detach(), reduction="none").sum(-1)
    kl_ba = F.kl_div(log_b, prob_a.detach(), reduction="none").sum(-1)
    T = a.shape[1]
    mask = torch.arange(T, device=a.device)[None, :] < valid_lens.to(a.device)[:, None]
    return ((kl_ab + kl_ba) * 0.5 * mask).sum() / mask.sum().clamp(min=1)


def compute_training_loss(model, batch, ctc_fn, epoch: int):
    sk, mo, y, input_lens, target_lens, _ = batch
    sk = sk.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)
    input_lens_gpu = input_lens.to(DEVICE, non_blocking=True)
    if MODEL_NAME == "pipeline":
        mo = mo.to(DEVICE, non_blocking=True)
        aux, pri = model(sk, mo)
        pri_loss = ctc_loss_for_logits(pri, y, input_lens, target_lens, ctc_fn)
        aux_loss = ctc_loss_for_logits(aux, y, input_lens, target_lens, ctc_fn)
        loss = pri_loss + CONFIG["aux_weight"] * aux_loss
        if CONFIG["lambda_kl"] > 0:
            loss = loss + CONFIG["lambda_kl"] * symmetric_kl(aux, pri, pooled_lengths(input_lens_gpu, pri.shape[1]), CONFIG["kl_temp"])
        return loss, pri

    outputs = model(sk)
    fuse = outputs[-1]
    fuse_loss = ctc_loss_for_logits(fuse, y, input_lens, target_lens, ctc_fn)
    stream_loss = sum(ctc_loss_for_logits(outputs[i], y, input_lens, target_lens, ctc_fn) for i in [0, 1, 2]) / 3.0
    gamma = min(CONFIG["stream_weight"], CONFIG["stream_weight"] * (epoch + 1) / max(1, CONFIG["stream_ramp_epochs"]))
    loss = fuse_loss + gamma * stream_loss
    if epoch >= CONFIG["distill_warmup"]:
        temp = CONFIG["distill_temp"]
        alpha = min(CONFIG["distill_max_weight"], (epoch - CONFIG["distill_warmup"] + 1) * CONFIG["distill_step"])
        with torch.no_grad():
            target = F.softmax(fuse.detach() / temp, dim=-1)
        distill = 0.0
        for i in range(4):
            distill = distill + F.kl_div(F.log_softmax(outputs[i] / temp, dim=-1), target, reduction="batchmean")
        loss = loss + alpha * (temp ** 2) * distill / 4.0
    return loss, fuse


def forward_logits(model, batch, head_index=-1):
    sk, mo, _, input_lens, _, _ = batch
    sk = sk.to(DEVICE, non_blocking=True)
    if MODEL_NAME == "pipeline":
        mo = mo.to(DEVICE, non_blocking=True)
        _, logits = model(sk, mo)
        return logits, input_lens
    outputs = model(sk)
    return outputs[head_index], input_lens


def edit_ops(ref_tokens, hyp_tokens):
    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    back = [[None] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        dp[i][0] = i
        back[i][0] = "D"
    for j in range(1, n + 1):
        dp[0][j] = j
        back[0][j] = "I"
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                choices = [(dp[i - 1][j - 1], "M")]
            else:
                choices = [(dp[i - 1][j - 1] + 1, "S")]
            choices.extend([(dp[i - 1][j] + 1, "D"), (dp[i][j - 1] + 1, "I")])
            dp[i][j], back[i][j] = min(choices, key=lambda x: x[0])
    i, j = m, n
    ops = []
    while i > 0 or j > 0:
        op = back[i][j]
        if op == "M":
            ops.append(("M", ref_tokens[i - 1], hyp_tokens[j - 1]))
            i -= 1
            j -= 1
        elif op == "S":
            ops.append(("S", ref_tokens[i - 1], hyp_tokens[j - 1]))
            i -= 1
            j -= 1
        elif op == "D":
            ops.append(("D", ref_tokens[i - 1], ""))
            i -= 1
        else:
            ops.append(("I", "", hyp_tokens[j - 1]))
            j -= 1
    ops.reverse()
    s = sum(1 for op, _, _ in ops if op == "S")
    d = sum(1 for op, _, _ in ops if op == "D")
    ins = sum(1 for op, _, _ in ops if op == "I")
    return {"S": s, "D": d, "I": ins, "N": max(1, m), "ops": ops}


def greedy_decode(log_probs, feat_lens, inv_vocab):
    best = log_probs.argmax(dim=-1).cpu()
    decoded = []
    for i in range(best.shape[0]):
        seq = best[i, : int(feat_lens[i])].tolist()
        out, prev = [], None
        for idx in seq:
            if idx != prev and idx != 0:
                out.append(inv_vocab.get(idx, ""))
            prev = idx
        decoded.append(" ".join(out))
    return decoded


def beam_decode_one(log_probs, inv_vocab, beam_size=5, topk=25):
    arr = log_probs.detach().cpu().numpy()
    beams = {(): (0.0, -np.inf)}
    vocab_size = arr.shape[1]
    topk = min(topk, vocab_size)
    for t in range(arr.shape[0]):
        ids = np.argpartition(arr[t], -topk)[-topk:]
        if 0 not in ids:
            ids = np.concatenate([[0], ids])
        next_beams = defaultdict(lambda: [-np.inf, -np.inf])
        for prefix, (pb, pnb) in beams.items():
            total = np.logaddexp(pb, pnb)
            next_beams[prefix][0] = np.logaddexp(next_beams[prefix][0], total + arr[t, 0])
            for c in ids:
                c = int(c)
                if c == 0:
                    continue
                p = arr[t, c]
                new_prefix = prefix + (c,)
                if prefix and c == prefix[-1]:
                    next_beams[new_prefix][1] = np.logaddexp(next_beams[new_prefix][1], pb + p)
                    next_beams[prefix][1] = np.logaddexp(next_beams[prefix][1], pnb + p)
                else:
                    next_beams[new_prefix][1] = np.logaddexp(next_beams[new_prefix][1], total + p)
        beams = dict(sorted(next_beams.items(), key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]), reverse=True)[:beam_size])
    best = max(beams, key=lambda k: np.logaddexp(beams[k][0], beams[k][1])) if beams else ()
    return " ".join(inv_vocab.get(int(c), "") for c in best)


def decode_batch(log_probs, feat_lens, inv_vocab, method: str):
    if method == "greedy":
        return greedy_decode(log_probs, feat_lens, inv_vocab)
    return [
        beam_decode_one(log_probs[i, : int(feat_lens[i])], inv_vocab, CONFIG["beam_width"], CONFIG["beam_topk"])
        for i in range(log_probs.shape[0])
    ]


@torch.no_grad()
def evaluate(model, loader, inv_vocab, split_name: str, decode_method="greedy", head_index=-1):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc=f"eval {split_name} {decode_method}", leave=False):
        logits, input_lens = forward_logits(model, batch, head_index=head_index)
        log_probs = logits.log_softmax(-1)
        feat_lens = pooled_lengths(input_lens, logits.shape[1])
        preds = decode_batch(log_probs, feat_lens, inv_vocab, decode_method)
        items = batch[-1]
        for item, pred in zip(items, preds):
            ref = item["gloss"]
            stats = edit_ops(ref.split(), pred.split())
            wer = (stats["S"] + stats["D"] + stats["I"]) / stats["N"]
            rows.append({
                "video": item["video"],
                "split": item["split"],
                "reference": ref,
                "prediction": pred,
                "wer": wer,
                "S": stats["S"],
                "D": stats["D"],
                "I": stats["I"],
                "N": stats["N"],
                "ref_len": len(ref.split()),
                "frames": item.get("frames", ""),
                "lh_detect_rate": item.get("lh_detect_rate", ""),
                "rh_detect_rate": item.get("rh_detect_rate", ""),
                "pose_detect_rate": item.get("pose_detect_rate", ""),
            })
    totals = Counter()
    for row in rows:
        totals["S"] += int(row["S"])
        totals["D"] += int(row["D"])
        totals["I"] += int(row["I"])
        totals["N"] += int(row["N"])
    wer = (totals["S"] + totals["D"] + totals["I"]) / max(1, totals["N"])
    metrics = {
        "split": split_name,
        "decode": decode_method,
        "wer": wer,
        "substitution_rate": totals["S"] / max(1, totals["N"]),
        "deletion_rate": totals["D"] / max(1, totals["N"]),
        "insertion_rate": totals["I"] / max(1, totals["N"]),
        "S": int(totals["S"]),
        "D": int(totals["D"]),
        "I": int(totals["I"]),
        "N": int(totals["N"]),
        "samples": len(rows),
    }
    return metrics, rows


def make_optimizer(model):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )


def make_scheduler(optimizer, total_epochs):
    warmup = CONFIG["warmup_epochs"]
    def lr_lambda(epoch):
        if warmup > 0 and epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(1, total_epochs - warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def set_backbone_trainable(model, trainable: bool):
    if MODEL_NAME == "pipeline":
        freeze_prefixes = ("body_", "lh_", "rh_", "face_", "mouth_")
    else:
        freeze_prefixes = ("enc.",)
    for name, param in model.named_parameters():
        if name.startswith(freeze_prefixes):
            param.requires_grad = trainable


def load_pretrained_for_finetune(model, ckpt_path: Path):
    if not ckpt_path or str(ckpt_path) in {"", "."}:
        print("No pretrained checkpoint path configured.")
        return
    if not ckpt_path.exists():
        print(f"Pretrained checkpoint not found: {ckpt_path}")
        return
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("model_state", ckpt.get("model", ckpt))
    current = model.state_dict()
    loaded, skipped = {}, []
    for key, value in state.items():
        clean_key = key.replace("module.", "")
        if clean_key in current and tuple(current[clean_key].shape) == tuple(value.shape):
            loaded[clean_key] = value
        else:
            skipped.append(clean_key)
    current.update(loaded)
    model.load_state_dict(current, strict=True)
    print(f"Loaded {len(loaded)} tensors from pretrained checkpoint.")
    print(f"Skipped {len(skipped)} tensors, usually classifier/vocab-specific heads.")
    if skipped:
        print("First skipped keys:", skipped[:10])


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_dev_wer, history):
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    torch.save({
        "experiment_name": EXPERIMENT_NAME,
        "model_name": MODEL_NAME,
        "stage": STAGE,
        "dataset_name": DATASET_NAME,
        "epoch": epoch,
        "model_state": raw_model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_dev_wer": best_dev_wer,
        "history": history,
        "vocab": vocab,
        "config": CONFIG,
        "keypoint_layout": "MediaPipe-86: pose[0:33], left_hand[33:54], right_hand[54:75], face[75:82], mouth[82:86]",
    }, path)


def train_model(model, train_loader, dev_loader):
    if STAGE == "finetune":
        load_pretrained_for_finetune(model, PRETRAIN_CKPT_PATH)
        if CONFIG["freeze_backbone_epochs"] > 0:
            set_backbone_trainable(model, False)
            print(f"Backbone frozen for first {CONFIG['freeze_backbone_epochs']} epochs.")

    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, CONFIG["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    ctc_fn = nn.CTCLoss(blank=0, zero_infinity=True)
    history = []
    best_dev_wer = float("inf")
    patience = 0

    for epoch in range(CONFIG["epochs"]):
        if STAGE == "finetune" and epoch == CONFIG["freeze_backbone_epochs"] and CONFIG["freeze_backbone_epochs"] > 0:
            set_backbone_trainable(model, True)
            optimizer = make_optimizer(model)
            scheduler = make_scheduler(optimizer, CONFIG["epochs"] - epoch)
            print("Backbone unfrozen. Optimizer reset for full fine-tuning.")

        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"epoch {epoch:03d}", leave=False)
        for step, batch in enumerate(pbar):
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                loss, _ = compute_training_loss(model, batch, ctc_fn, epoch)
                loss = loss / CONFIG["accum_steps"]
            scaler.scale(loss).backward()
            if (step + 1) % CONFIG["accum_steps"] == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running_loss += float(loss.item()) * CONFIG["accum_steps"]
            pbar.set_postfix(loss=f"{running_loss / max(1, step + 1):.4f}")

        scheduler.step()
        train_loss = running_loss / max(1, len(train_loader))
        dev_metrics, _ = evaluate(model, dev_loader, inv_vocab, "dev", decode_method="greedy")
        current_lr = optimizer.param_groups[0]["lr"]
        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_wer": dev_metrics["wer"],
            "dev_substitution_rate": dev_metrics["substitution_rate"],
            "dev_deletion_rate": dev_metrics["deletion_rate"],
            "dev_insertion_rate": dev_metrics["insertion_rate"],
            "lr": current_lr,
        }
        history.append(record)
        save_csv(history, OUTPUT_DIR / "training_log.csv")
        print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | dev WER={dev_metrics['wer']:.4f} | lr={current_lr:.2e}")

        save_checkpoint(OUTPUT_DIR / "last_model.pt", model, optimizer, scheduler, epoch, best_dev_wer, history)
        if dev_metrics["wer"] < best_dev_wer:
            best_dev_wer = dev_metrics["wer"]
            patience = 0
            save_checkpoint(OUTPUT_DIR / "best_model.pt", model, optimizer, scheduler, epoch, best_dev_wer, history)
            print(f"Saved best model with Dev WER={best_dev_wer:.4f}")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print(f"Early stopping at epoch {epoch}. Best Dev WER={best_dev_wer:.4f}")
                break
    return history


## Plotting and report artifacts


In [6]:
def plot_history(history):
    if not history:
        return
    epochs = [r["epoch"] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs, [r["train_loss"] for r in history], marker="o")
    axes[0].set_title("Training CTC Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, [r["dev_wer"] for r in history], marker="o", color="tab:red")
    axes[1].set_title("Dev WER")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, [r["lr"] for r in history], marker="o", color="tab:green")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=220)
    plt.close()


def plot_error_breakdown(metrics, name):
    labels = ["Substitution", "Deletion", "Insertion"]
    values = [
        metrics["substitution_rate"] * 100,
        metrics["deletion_rate"] * 100,
        metrics["insertion_rate"] * 100,
    ]
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, values, color=["#4C78A8", "#F58518", "#54A24B"])
    ax.bar_label(bars, fmt="%.1f%%")
    ax.set_ylabel("Rate over reference words (%)")
    ax.set_title(f"{name} WER Breakdown")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name.lower()}_wer_breakdown.png", dpi=220)
    plt.close()


def aggregate_bucket(rows, key_fn):
    buckets = defaultdict(lambda: Counter())
    for row in rows:
        key = key_fn(row)
        buckets[key]["S"] += int(row["S"])
        buckets[key]["D"] += int(row["D"])
        buckets[key]["I"] += int(row["I"])
        buckets[key]["N"] += int(row["N"])
    out = []
    for key, c in sorted(buckets.items(), key=lambda kv: str(kv[0])):
        wer = (c["S"] + c["D"] + c["I"]) / max(1, c["N"])
        out.append({"bucket": key, "wer": wer, "N": int(c["N"]), "S": int(c["S"]), "D": int(c["D"]), "I": int(c["I"])})
    return out


def length_bucket(row):
    n = int(row["ref_len"])
    if n <= 4:
        return "short_2_4"
    if n <= 7:
        return "medium_5_7"
    return "long_8_plus"


def hand_quality_bucket(row):
    vals = []
    for key in ["lh_detect_rate", "rh_detect_rate"]:
        try:
            vals.append(float(row[key]))
        except Exception:
            pass
    q = min(vals) if vals else 0.0
    if q < 20:
        return "hand_lt_20"
    if q < 40:
        return "hand_20_40"
    if q < 60:
        return "hand_40_60"
    return "hand_60_plus"


def plot_bucket_wer(rows, key_fn, filename, title):
    data = aggregate_bucket(rows, key_fn)
    save_csv(data, OUTPUT_DIR / filename.replace(".png", ".csv"))
    if not data:
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar([d["bucket"] for d in data], [d["wer"] * 100 for d in data], color="#4C78A8")
    ax.bar_label(bars, fmt="%.1f%%")
    ax.set_ylabel("WER (%)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=220)
    plt.close()


def save_top_error_tables(rows):
    substitutions = Counter()
    deletions = Counter()
    insertions = Counter()
    for row in rows:
        ref = row["reference"].split()
        hyp = row["prediction"].split()
        for op, r, h in edit_ops(ref, hyp)["ops"]:
            if op == "S":
                substitutions[(r, h)] += 1
            elif op == "D":
                deletions[r] += 1
            elif op == "I":
                insertions[h] += 1
    save_csv(
        [{"ref": k[0], "pred": k[1], "count": v} for k, v in substitutions.most_common(50)],
        OUTPUT_DIR / "top_substitutions.csv",
        ["ref", "pred", "count"],
    )
    save_csv(
        [{"ref": k, "count": v} for k, v in deletions.most_common(50)],
        OUTPUT_DIR / "top_deletions.csv",
        ["ref", "count"],
    )
    save_csv(
        [{"pred": k, "count": v} for k, v in insertions.most_common(50)],
        OUTPUT_DIR / "top_insertions.csv",
        ["pred", "count"],
    )


def plot_mska_stream_wers(stream_metrics):
    if not stream_metrics:
        return
    names = list(stream_metrics.keys())
    values = [stream_metrics[n]["wer"] * 100 for n in names]
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(names, values, color=["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756"])
    ax.bar_label(bars, fmt="%.1f%%")
    ax.set_ylabel("WER (%)")
    ax.set_title("MSKA Per-Stream Test WER")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "mska_stream_wer.png", dpi=220)
    plt.close()


## Run experiment

Final evaluation is performed on the test split. The notebook saves prediction CSV files for qualitative inspection.


In [7]:
model = create_model(len(vocab)).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

def print_model_summary(model):
    summary = {
        "Experiment": EXPERIMENT_NAME,
        "Model": model.__class__.__name__,
        "Stage": STAGE,
        "Dataset": DATASET_NAME,
        "Run mode": RUN_MODE,
        "Vocab size": len(vocab),
        "Input layout": "(T, 86, 3) MediaPipe skeleton",
        "Parameters": f"{num_params:,}",
        "Trainable parameters": f"{trainable_params:,}",
        "Output dir": str(OUTPUT_DIR),
    }
    print("Model summary")
    for key, value in summary.items():
        print(f"  {key:22s}: {value}")


def metric_row(label, metrics):
    return {
        "Metric": label,
        "WER (%)": round(metrics["wer"] * 100, 2),
        "Sub (%)": round(metrics["substitution_rate"] * 100, 2),
        "Del (%)": round(metrics["deletion_rate"] * 100, 2),
        "Ins (%)": round(metrics["insertion_rate"] * 100, 2),
        "S": metrics["S"],
        "D": metrics["D"],
        "I": metrics["I"],
        "N": metrics["N"],
        "Samples": metrics["samples"],
    }


print_model_summary(model)

start_time = time.time()
history = train_model(model, train_loader, dev_loader)
plot_history(history)

best_path = OUTPUT_DIR / "best_model.pt"
best_epoch = None
best_dev_wer = None
if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    best_epoch = ckpt.get("epoch")
    best_dev_wer = ckpt.get("best_dev_wer")
    print(f"Loaded best checkpoint from epoch {ckpt['epoch']} with Dev WER={ckpt['best_dev_wer']:.4f}")

dev_metrics, dev_rows = evaluate(model, dev_loader, inv_vocab, "dev", decode_method="greedy")
test_greedy_metrics, test_greedy_rows = evaluate(model, test_loader, inv_vocab, "test", decode_method="greedy")
if RUN_MODE == "smoke" and not CONFIG["run_beam_in_smoke"]:
    print("Skipping CTC beam search in smoke mode. Greedy decode is enough for sanity check.")
    test_beam_metrics, test_beam_rows = None, []
else:
    test_beam_metrics, test_beam_rows = evaluate(model, test_loader, inv_vocab, "test", decode_method="beam")

save_csv(dev_rows, OUTPUT_DIR / "predictions_dev_greedy.csv")
save_csv(test_greedy_rows, OUTPUT_DIR / "predictions_test_greedy.csv")
if test_beam_rows:
    save_csv(test_beam_rows, OUTPUT_DIR / "predictions_test_beam.csv")

final_metrics = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "stage": STAGE,
    "dataset_name": DATASET_NAME,
    "run_mode": RUN_MODE,
    "parameters_total": int(num_params),
    "parameters_trainable_initial": int(trainable_params),
    "elapsed_minutes": (time.time() - start_time) / 60.0,
    "best_epoch": best_epoch,
    "best_dev_wer": best_dev_wer,
    "dev_greedy": dev_metrics,
    "test_greedy": test_greedy_metrics,
    "test_beam": test_beam_metrics,
}

if MODEL_NAME == "mska":
    stream_metrics = {}
    for idx, name in enumerate(MSKA_STREAM_NAMES):
        m, _ = evaluate(model, test_loader, inv_vocab, "test", decode_method="greedy", head_index=idx)
        stream_metrics[name] = m
    final_metrics["mska_streams_test_greedy"] = stream_metrics
    save_json(stream_metrics, OUTPUT_DIR / "mska_stream_wers.json")
    plot_mska_stream_wers(stream_metrics)

save_json(final_metrics, OUTPUT_DIR / "metrics.json")
report_metrics = test_beam_metrics if test_beam_metrics is not None else test_greedy_metrics
report_rows = test_beam_rows if test_beam_rows else test_greedy_rows
report_name = "Test_Beam" if test_beam_metrics is not None else "Test_Greedy"
plot_error_breakdown(report_metrics, report_name)
plot_bucket_wer(report_rows, length_bucket, "test_wer_by_sentence_length.png", "Test WER by Sentence Length")
plot_bucket_wer(report_rows, hand_quality_bucket, "test_wer_by_hand_quality.png", "Test WER by Hand Keypoint Quality")
save_top_error_tables(report_rows)

summary_rows = [
    metric_row("Dev Greedy", dev_metrics),
    metric_row("Test Greedy", test_greedy_metrics),
]
if test_beam_metrics is not None:
    summary_rows.append(metric_row("Test Beam", test_beam_metrics))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "final_metrics_summary.csv", index=False)
print("Final evaluation summary")
display(summary_df)
print("Artifacts saved to:", OUTPUT_DIR)


Model summary
  Experiment            : phoenix14t_pipeline_pretrain
  Model                 : PipelineCSLRModel
  Stage                 : pretrain
  Dataset               : PHOENIX14T
  Run mode              : full
  Vocab size            : 1116
  Input layout          : (T, 86, 3) MediaPipe skeleton
  Parameters            : 4,845,800
  Trainable parameters  : 4,845,800
  Output dir            : /kaggle/working/phoenix14t_pipeline_pretrain


epoch 000:   0%|          | 0/887 [00:00<?, ?it/s]

eval dev greedy:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 000 | loss=11.1353 | dev WER=1.0000 | lr=1.60e-04
Saved best model with Dev WER=1.0000


epoch 001:   0%|          | 0/887 [00:00<?, ?it/s]

...

eval test greedy:   0%|          | 0/81 [00:00<?, ?it/s]

eval test beam:   0%|          | 0/81 [00:00<?, ?it/s]

Final evaluation summary


,Metric,WER (%),Sub (%),Del (%),Ins (%),S,D,I,N,Samples
0,Dev Greedy,32.55,21.00,7.79,3.76,787,292,141,3748,519
1,Test Greedy,33.47,22.49,6.87,4.10,959,293,175,4264,642
2,Test Beam,33.23,22.42,6.80,4.01,956,290,171,4264,642


Artifacts saved to: /kaggle/working/phoenix14t_pipeline_pretrain
